# Dalle feature all'apprendimento

**Computer Vision — Arcangelo Distante e Cosimo Distante**

Materiali del capitolo 7. Il notebook contiene tre esperimenti analitici con dati sintetici: scala delle feature e vicini, score e loss, conteggi e metriche. Le quantità sono costruzioni didattiche riproducibili; non sono risultati di un benchmark di immagini.

L'esecuzione richiede **CPU e NumPy**. Non servono immagini, pesi, credenziali o altri file. Le attività N7.A1–N7.A3 invitano a interpretare e modificare gli esperimenti. Le soluzioni complete sono nel materiale riservato al docente.


## Avvio locale e in Colab

In Colab aprire questo file con **File → Apri notebook → Carica**, oppure tramite il collegamento del repository del libro. In **Runtime → Cambia tipo di runtime** scegliere Python 3 con CPU. Se NumPy non fosse disponibile, eseguire `%pip install numpy` in una cella prima degli import. Tutti i dati sono definiti qui e quindi appartengono allo stesso runtime.

In locale usare Python 3 con NumPy installato e aprire il notebook in Jupyter. Eseguire le celle nell'ordine. Dopo un riavvio, ripartire dall'import e ricreare dati e funzioni. I risultati vengono salvati nella cartella `output_capitolo07`; al termine si possono scaricare i file e il notebook con output. L'esecuzione locale e quella nel servizio Colab sono ambienti distinti: i tempi di uno non descrivono l'altro.


In [1]:
from pathlib import Path
import csv
import json
import platform
import sys
import numpy as np

np.set_printoptions(precision=6, suppress=True)
OUTPUT = Path("output_capitolo07")
OUTPUT.mkdir(exist_ok=True)
risultati = {
    "natura_dati": "Costruzioni analitiche sintetiche, non benchmark di immagini",
    "dispositivo": "CPU",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "runtime": "Google Colab" if "google.colab" in sys.modules else "Python locale o altro runtime",
}
print("Python", risultati["python"], "NumPy", risultati["numpy"])
print(risultati["natura_dati"])


Python 3.12.14 NumPy 2.3.5
Costruzioni analitiche sintetiche, non benchmark di immagini


## Esperimento 1 — Scala e vicino più prossimo

Due punti di training illustrano il calcolo della distanza; non permettono di stimare una generalizzazione significativa. La prima coordinata e la seconda hanno scale numeriche differenti. La standardizzazione usa media e deviazione standard del solo training, con denominatore $N$ (`ddof=0`). La query usa gli stessi parametri.


In [2]:
def fit_standardizzazione(X):
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2 or X.shape[0] == 0:
        raise ValueError("Il training deve essere una matrice non vuota.")
    media = X.mean(axis=0)
    scala = X.std(axis=0, ddof=0)
    # Una coordinata costante non può essere divisa per zero.
    scala = np.where(scala == 0, 1.0, scala)
    return media, scala

def trasforma(X, media, scala):
    return (np.asarray(X, dtype=np.float64) - media) / scala

def knn_predici(X_train, y_train, query, k=1):
    X_train = np.asarray(X_train, dtype=np.float64)
    y_train = np.asarray(y_train, dtype=np.int64)
    query = np.atleast_2d(np.asarray(query, dtype=np.float64))
    if not (1 <= k <= len(X_train)):
        raise ValueError("k deve essere fra 1 e la numerosità del training.")
    if X_train.ndim != 2 or X_train.shape[1] != query.shape[1]:
        raise ValueError("Training e query devono avere la stessa dimensione di feature.")
    if len(y_train) != len(X_train) or np.any(y_train < 0):
        raise ValueError("Servono etichette intere non negative per ogni riga.")
    distanze = np.sqrt(((query[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2))
    # Pareggi fra distanze: ordine originale dei punti.
    indici = np.argsort(distanze, axis=1, kind="stable")[:, :k]
    # Pareggi fra voti: classe con indice minore.
    predizioni = np.array([np.bincount(y_train[riga]).argmax() for riga in indici])
    return predizioni, distanze, indici

X_train = np.array([[0.0, 1000.0], [1.0, 0.0]])
y_train = np.array([0, 1])
query = np.array([[0.9, 800.0]])
media, scala = fit_standardizzazione(X_train)
X_standard = trasforma(X_train, media, scala)
query_standard = trasforma(query, media, scala)
pred_grezza, dist_grezze, _ = knn_predici(X_train, y_train, query)
pred_standard, dist_standard, _ = knn_predici(X_standard, y_train, query_standard)

print("Media training:", media, "Scala training:", scala)
print("Training standardizzato:\n", X_standard)
print("Query standardizzata:", query_standard)
print("Distanze grezze:", dist_grezze, "Classe:", pred_grezza)
print("Distanze standardizzate:", dist_standard, "Classe:", pred_standard)

risultati["scala_knn"] = {
    "training": X_train.tolist(), "etichette_training": y_train.tolist(),
    "query": query.tolist(), "media": media.tolist(), "scala": scala.tolist(),
    "distanze_grezze": dist_grezze.tolist(), "predizione_grezza": pred_grezza.tolist(),
    "distanze_standardizzate": dist_standard.tolist(),
    "predizione_standardizzata": pred_standard.tolist(),
}


Media training: [  0.5 500. ] Scala training: [  0.5 500. ]
Training standardizzato:
 [[-1.  1.]
 [ 1. -1.]]
Query standardizzata: [[0.8 0.6]]
Distanze grezze: [[200.002025 800.000006]] Classe: [0]
Distanze standardizzate: [[1.843909 1.612452]] Classe: [1]


In [3]:
# Cambio comune di unità nella seconda coordinata.
fattori_unita = np.array([1.0, 0.001])
X_convertito = X_train * fattori_unita
query_convertita = query * fattori_unita
media_convertita, scala_convertita = fit_standardizzazione(X_convertito)
X_standard_convertito = trasforma(X_convertito, media_convertita, scala_convertita)
query_standard_convertita = trasforma(query_convertita, media_convertita, scala_convertita)
pred_convertita, dist_convertite, _ = knn_predici(X_convertito, y_train, query_convertita)

print("Distanze dopo cambio di unità:", dist_convertite)
print("Predizione grezza dopo cambio di unità:", pred_convertita)
print("Differenza massima fra training standardizzati:", np.max(np.abs(X_standard - X_standard_convertito)))
print("Differenza massima fra query standardizzate:", np.max(np.abs(query_standard - query_standard_convertita)))


Distanze dopo cambio di unità: [[0.921954 0.806226]]
Predizione grezza dopo cambio di unità: [1]
Differenza massima fra training standardizzati: 0.0
Differenza massima fra query standardizzate: 1.1102230246251565e-16


### Attività N7.A1

1. **N7.A1.1** Identificare il vicino prima e dopo standardizzazione. Spiegare il cambiamento usando i contributi delle due coordinate.
2. **N7.A1.2** Interpretare il cambio di unità: perché può cambiare il vicino grezzo e perché, in questo caso, le coordinate standardizzate restano uguali? Provare un altro fattore positivo comune alla seconda coordinata di training e query.
3. **N7.A1.3** Indicare quali dati possono entrare nel fit di media e scala. Spiegare che cosa cambierebbe includendo le query in tale fit.


## Esperimento 2 — Score, softmax, temperatura e loss

La softmax è calcolata sottraendo il massimo. La cross-entropy usa log-sum-exp e riceve gli score prima della softmax. La temperatura è assegnata: questo esperimento non stima una calibrazione da etichette osservate.


In [4]:
def softmax_stabile(score, temperatura=1.0):
    if temperatura <= 0:
        raise ValueError("La temperatura deve essere positiva.")
    score = np.asarray(score, dtype=np.float64) / temperatura
    relativi = score - np.max(score, axis=-1, keepdims=True)
    esponenziali = np.exp(relativi)
    return esponenziali / esponenziali.sum(axis=-1, keepdims=True)

def cross_entropy_score(score, target):
    score = np.asarray(score, dtype=np.float64)
    massimo = np.max(score)
    return float(-score[target] + massimo + np.log(np.exp(score - massimo).sum()))

def hinge_multiclasse(score, target, margine=1.0):
    score = np.asarray(score, dtype=np.float64)
    contributi = np.maximum(0.0, score - score[target] + margine)
    contributi[target] = 0.0
    return float(contributi.sum())

score = np.array([2.0, 1.0, 0.0])
score_traslati = score + 1000.0
probabilita = softmax_stabile(score)
probabilita_traslate = softmax_stabile(score_traslati)
temperature = [0.5, 1.0, 2.0]
tabella_temperature = {str(T): softmax_stabile(score, T).tolist() for T in temperature}
ce = cross_entropy_score(score, target=0)
hinge = hinge_multiclasse(score, target=0)

print("Score:", score, "Probabilità:", probabilita)
print("Score traslati:", score_traslati, "Probabilità:", probabilita_traslate)
print("Differenza massima:", np.max(np.abs(probabilita - probabilita_traslate)))
for T in temperature:
    p = softmax_stabile(score, T)
    print("T =", T, "p =", p, "argmax =", int(p.argmax()))
print("Target 0: cross-entropy =", ce, "hinge =", hinge)
print("Derivata CE rispetto agli score:", probabilita - np.array([1.0, 0.0, 0.0]))

risultati["softmax_loss"] = {
    "score": score.tolist(), "probabilita": probabilita.tolist(),
    "probabilita_score_traslati": probabilita_traslate.tolist(),
    "temperature": tabella_temperature,
    "cross_entropy_target0": ce, "hinge_target0_margine1": hinge,
}


Score: [2. 1. 0.] Probabilità: [0.665241 0.244728 0.090031]
Score traslati: [1002. 1001. 1000.] Probabilità: [0.665241 0.244728 0.090031]
Differenza massima: 0.0
T = 0.5 p = [0.866813 0.11731  0.015876] argmax = 0
T = 1.0 p = [0.665241 0.244728 0.090031] argmax = 0
T = 2.0 p = [0.50648  0.307196 0.186324] argmax = 0
Target 0: cross-entropy = 0.4076059644443804 hinge = 0.0
Derivata CE rispetto agli score: [-0.334759  0.244728  0.090031]


### Attività N7.A2

1. **N7.A2.1** Motivare l'uguaglianza delle probabilità per score traslati di 1000. Spiegare il ruolo della sottrazione del massimo.
2. **N7.A2.2** Confrontare concentrazione e argmax alle tre temperature. Indicare quale informazione manchi per giudicare la calibrazione su immagini reali.
3. **N7.A2.3** Interpretare il caso con previsione corretta, hinge nulla e cross-entropy positiva. Costruire un altro vettore di score che mantenga l'argmax e cambi la cross-entropy.


## Esperimento 3 — Matrice di confusione e medie

Le coppie di etichette sono costruite a partire da conteggi assegnati. Le righe indicano la classe vera e le colonne la classe predetta. Per i denominatori nulli si usa la convenzione esplicita `zero_division=0` e si restituiscono anche gli indicatori delle quantità definite. La convenzione consente il calcolo di una sintesi, ma non sostituisce osservazioni mancanti.


In [5]:
def matrice_confusione(y_vera, y_predetta, n_classi):
    y_vera = np.asarray(y_vera, dtype=np.int64)
    y_predetta = np.asarray(y_predetta, dtype=np.int64)
    if y_vera.shape != y_predetta.shape or y_vera.ndim != 1:
        raise ValueError("Le etichette devono essere vettori della stessa lunghezza.")
    if np.any(y_vera < 0) or np.any(y_vera >= n_classi) or np.any(y_predetta < 0) or np.any(y_predetta >= n_classi):
        raise ValueError("Etichette fuori dall'insieme delle classi dichiarate.")
    M = np.zeros((n_classi, n_classi), dtype=np.int64)
    np.add.at(M, (y_vera, y_predetta), 1)
    return M

def rapporto_zero(num, den):
    num = np.asarray(num, dtype=np.float64)
    den = np.asarray(den, dtype=np.float64)
    return np.divide(num, den, out=np.zeros_like(num), where=den != 0)

def metriche_da_conteggi(M):
    M = np.asarray(M, dtype=np.int64)
    tp = np.diag(M)
    supporto = M.sum(axis=1)
    predette = M.sum(axis=0)
    fp = predette - tp
    fn = supporto - tp
    precision = rapporto_zero(tp, predette)
    recall = rapporto_zero(tp, supporto)
    f1 = rapporto_zero(2 * tp, 2 * tp + fp + fn)
    totale = int(M.sum())
    if totale == 0:
        raise ValueError("Serve almeno una osservazione per una valutazione.")
    tp_tot = int(tp.sum())
    fp_tot = int(fp.sum())
    fn_tot = int(fn.sum())
    micro_f1 = 2 * tp_tot / (2 * tp_tot + fp_tot + fn_tot)
    return {
        "matrice": M.tolist(), "supporto": supporto.tolist(),
        "precision": precision.tolist(), "recall": recall.tolist(), "f1": f1.tolist(),
        "precision_definita": (predette != 0).tolist(),
        "recall_definito": (supporto != 0).tolist(),
        "accuracy": tp_tot / totale, "macro_f1": float(f1.mean()),
        "micro_f1": float(micro_f1),
        "weighted_f1": float(np.dot(f1, supporto) / totale),
        "zero_division": 0,
    }

M_assegnata = np.array([[8, 1, 1], [2, 5, 3], [0, 2, 8]])
y_vera = []
y_predetta = []
for vera in range(3):
    for predetta in range(3):
        conteggio = int(M_assegnata[vera, predetta])
        y_vera.extend([vera] * conteggio)
        y_predetta.extend([predetta] * conteggio)
y_vera = np.array(y_vera)
y_predetta = np.array(y_predetta)
M = matrice_confusione(y_vera, y_predetta, n_classi=3)
report = metriche_da_conteggi(M)

print("Matrice ricostruita:\n", M)
for nome in ["supporto", "precision", "recall", "f1", "accuracy", "macro_f1", "micro_f1"]:
    print(nome, report[nome])
risultati["metriche"] = report


Matrice ricostruita:
 [[8 1 1]
 [2 5 3]
 [0 2 8]]
supporto [10, 10, 10]
precision [0.8, 0.625, 0.6666666666666666]
recall [0.8, 0.5, 0.8]
f1 [0.8, 0.5555555555555556, 0.7272727272727273]
accuracy 0.7
macro_f1 0.6942760942760943
micro_f1 0.7


In [6]:
predizioni_costanti = np.zeros_like(y_vera)
M_costante = matrice_confusione(y_vera, predizioni_costanti, n_classi=3)
report_costante = metriche_da_conteggi(M_costante)
print("Predizione costante classe 0:\n", M_costante)
for nome in ["precision", "precision_definita", "recall", "f1", "accuracy", "macro_f1", "micro_f1"]:
    print(nome, report_costante[nome])
risultati["metriche_predizione_costante"] = report_costante


Predizione costante classe 0:
 [[10  0  0]
 [10  0  0]
 [10  0  0]]
precision [0.3333333333333333, 0.0, 0.0]
precision_definita [True, False, False]
recall [1.0, 0.0, 0.0]
f1 [0.5, 0.0, 0.0]
accuracy 0.3333333333333333
macro_f1 0.16666666666666666
micro_f1 0.3333333333333333


### Attività N7.A3

1. **N7.A3.1** Ricostruire a mano precision, recall e F1 della classe 1. Confrontare macro-F1, micro-F1 e accuracy, specificando le condizioni dell'identità fra le ultime due.
2. **N7.A3.2** Interpretare il classificatore che predice sempre 0. Indicare quali precision siano indefinite prima della convenzione e spiegare il valore della macro-F1 riportata.
3. **N7.A3.3** Descrivere che cosa andrebbe perso riportando soltanto l'accuracy. Proporre una sintesi adatta a un'applicazione in cui recuperare la classe 2 abbia particolare importanza.


## Salvataggio dei risultati

Il JSON conserva ambiente, dati analitici e risultati; il CSV riporta le metriche per classe dell'esempio principale. Salvare anche il notebook con gli output permette di associare i calcoli alle spiegazioni. In Colab il contenuto del runtime è temporaneo: scaricare i file prima di terminarlo o copiarli in una destinazione scelta dall'utente.


In [7]:
json_path = OUTPUT / "risultati_capitolo07.json"
csv_path = OUTPUT / "metriche_per_classe.csv"
json_path.write_text(json.dumps(risultati, indent=2, ensure_ascii=False), encoding="utf-8")
with csv_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["classe", "supporto", "precision", "recall", "f1"])
    for classe in range(3):
        writer.writerow([classe, report["supporto"][classe], report["precision"][classe], report["recall"][classe], report["f1"][classe]])
print("Salvati:", json_path, csv_path)


Salvati: output_capitolo07/risultati_capitolo07.json output_capitolo07/metriche_per_classe.csv


Per scaricare in Colab si può usare il pannello **File** del runtime, aprire `output_capitolo07` e scegliere **Scarica** per ciascun file. Salvare il notebook da **File → Scarica → Scarica .ipynb**. In locale i file sono nella cartella di lavoro del notebook.

Riferimenti teorici e letture sono indicati nel capitolo 7 e nella bibliografia del libro. Le costruzioni numeriche di questo notebook illustrano formule e convenzioni; una valutazione su immagini richiede dataset, split, training e condizioni di generalizzazione appropriati.
